# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 07: Production A/B Testing: Frequentist vs Bayesian Paradigms

### 1. Conceptual Intuition & Theoretical Background
Online A/B testing is the gold standard for measuring ML model improvements against production control.

**1. Frequentist Framework (Two-Proportion Z-Test):**
Tests null hypothesis $H_0: p_B = p_A$.
$$Z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$
- **Failure Mode ("Peeking Problem"):** Checking p-values continuously inflates false positive rate (Type I error) from $5\%$ to over $30\%$ without sequential testing corrections (e.g. O'Brien-Fleming).

**2. Bayesian Framework (Beta-Binomial Conjugate Updating):**
Models conversion rates as probability distributions. With prior $\text{Beta}(\alpha_0, \beta_0)$ and observing $k$ conversions in $n$ trials:
$$p \mid \text{Data} \sim \text{Beta}(\alpha_0 + k,\ \beta_0 + n - k)$$
Computes actionable metrics directly:
$$P(p_B > p_A) = \int_0^1 \int_{p_A}^1 f(p_A) f(p_B) dp_B dp_A$$

---

### 2. Scratch Python Implementation: Continuous Decision Engine

In [ ]:
import numpy as np

# True parameters (Variant B has a small true lift)
p_A_true = 0.050 # 5.0%
p_B_true = 0.058 # 5.8% (16% relative lift)

# Beta-Binomial Bayesian Updater
class BayesianABTest:
    def __init__(self, alpha_prior=1, beta_prior=1):
        self.a_A, self.b_A = alpha_prior, beta_prior
        self.a_B, self.b_B = alpha_prior, beta_prior
        
    def update(self, conv_A, no_conv_A, conv_B, no_conv_B):
        self.a_A += conv_A
        self.b_A += no_conv_A
        self.a_B += conv_B
        self.b_B += no_conv_B
        
    def compute_probability_B_beats_A(self, n_mc=10000):
        samples_A = np.random.beta(self.a_A, self.b_A, n_mc)
        samples_B = np.random.beta(self.a_B, self.b_B, n_mc)
        prob = np.mean(samples_B > samples_A)
        expected_lift = np.mean((samples_B - samples_A) / samples_A) * 100
        return prob, expected_lift

# Simulate 5000 users per variant
np.random.seed(42)
n_users = 5000
traffic_A = np.random.binomial(1, p_A_true, n_users)
traffic_B = np.random.binomial(1, p_B_true, n_users)

test = BayesianABTest()
test.update(conv_A=traffic_A.sum(), no_conv_A=n_users - traffic_A.sum(),
            conv_B=traffic_B.sum(), no_conv_B=n_users - traffic_B.sum())

prob_win, exp_lift = test.compute_probability_B_beats_A()

print(f"Observed Variant A Conversion: {traffic_A.mean()*100:.2f}%")
print(f"Observed Variant B Conversion: {traffic_B.mean()*100:.2f}%")
print(f"Probability Variant B > Variant A: {prob_win * 100:.2f}%")
print(f"Expected Relative Lift:             +{exp_lift:.2f}%")

### 3. Step-by-Step Code Walkthrough
1. **Conjugacy**: Beta prior + Binomial likelihood yields an exact analytical Beta posterior.
2. **Monte Carlo Integration**: Generates samples to answer questions directly (e.g. $P(B > A) = 99.1\%$) rather than abstract rejection of null hypotheses.
3. **Immunity to Peeking**: Can be updated and monitored continuously without inflating false positive rates.

---

### 4. Industry Applications & Trade-offs
- **E-Commerce Recommender Rollouts:** Early stopping of underperforming variants prevents lost sales revenue.
- **Trade-off:** Requires selecting sensible priors when sample sizes are small.

---

### 5. Top Interview Questions & Answers
**Q1: What is Sample Ratio Mismatch (SRM) and why does it invalidate an A/B test?**
> **Answer:** SRM occurs when the observed ratio of traffic between variants deviates from the expected design (e.g. expected 50/50, observed 55/45). It indicates hidden systematic bias (e.g., bot filtering, redirection crashes) that invalidates all causal conclusions.

**Q2: What is the "Peeking Problem" in frequentist testing?**
> **Answer:** Continuously evaluating p-values and stopping the experiment as soon as $p < 0.05$ invalidates the Type I error guarantee, increasing the false discovery rate to $30\%$ or higher.